## Import

This notebook mirrors the local inference workflow, but uses Azure OpenAI for single-query checking and batch input construction. Keep API credentials in `configs/API.key.yaml`; do not paste keys into notebook cells.


In [ ]:
from pathlib import Path
from pprint import pprint

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "main.py").exists():
        PROJECT_ROOT = candidate
        break
else:
    PROJECT_ROOT = Path.cwd().resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
from model.azure import (
    create_azure_client,
    azure_query_single,
    azure_query_batch_from_task,
    azure_query_batch_from_file,
    azure_query_batch_from_data,
    merge_azure_query_batch,
    parse_azure_result_batch,
    transform_azure_result_to_task_result,
    cost_estimation_word,
)


In [ ]:
from dataset.task import get_cls_for_all_tasks, get_type_task
from util.tool import load_json
from util.runtime import parse_prompt_mode


## Args

Configure model, task, prompt mode, and Azure credential provider here. The expected key file is `configs/API.key.yaml` with a top-level `AZURE-OPENAI` block containing `endpoint`, `api_version`, and `api_key`.

Required standard task fields: `id`, `split`, `language`, `type`, `instruction`, `input`, and `output`. Supported prompt modes are `direct`, `cot`, `direct-N-shot`, and `cot-N-shot`.


In [ ]:
class EmptyArgs:
    def __init__(self):
        self.dir_dataset = ["dataset_raw"]
        self.dir_result = ["result"]
        self.dir_result_save = self.dir_result[0]
        self.seed = 42
        self.sample_eval_index = 0


args = EmptyArgs()
for k, v in vars(args).items():
    print(f"{k}: {v}")


In [ ]:
path_file_api_key = PROJECT_ROOT / "configs" / "API.key.yaml"
api_provider = "AZURE-OPENAI"

if not path_file_api_key.exists():
    raise FileNotFoundError(f"API key file not found: {path_file_api_key}")

with path_file_api_key.open("r", encoding="utf-8") as f:
    dict_api_key = yaml.safe_load(f)

if api_provider not in dict_api_key:
    raise KeyError(f"Provider {api_provider!r} not found in {path_file_api_key}")

azure_config = dict_api_key[api_provider]
required_key_fields = ["endpoint", "api_version", "api_key"]
missing_key_fields = [field for field in required_key_fields if not azure_config.get(field)]
if missing_key_fields:
    raise ValueError(f"Missing Azure credential field(s): {missing_key_fields}")

AZURE_OPENAI_ENDPOINT = azure_config["endpoint"]
API_VERSION = azure_config["api_version"]
AZURE_OPENAI_API_KEY = azure_config["api_key"]

print("api_provider:", api_provider)
print("api_version:", API_VERSION)
print("endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("api_key loaded:", bool(AZURE_OPENAI_API_KEY))


In [ ]:
# Model/deployment name configured in Azure OpenAI.
model_name = "gpt-4o"
mode = None  # None = auto, or set to "chat" / "response"
prompt_mode = "direct"
split = "test"
max_token_input = 100 * 1024
max_token_output = 8192

print("model_name:", model_name)
print("prompt_mode:", parse_prompt_mode(prompt_mode))
print("split:", split)


## Task class

Select a benchmark task and instantiate the registered task adapter. Keep this separate from data loading so task registration problems are easy to diagnose.


In [ ]:
DICT_TYPE_TASK_CLS, DICT_TASK_CLS = get_cls_for_all_tasks()
print(f"Number of registered tasks: {len(DICT_TASK_CLS)}")
list(DICT_TASK_CLS.keys())[:10]


In [ ]:
task_name = "ADE-Identification"
if task_name not in DICT_TASK_CLS:
    raise KeyError(f"Task not registered: {task_name}")

task_cls = DICT_TASK_CLS[task_name]
task = task_cls(args=args, task=task_name)
task_type = get_type_task(task)
print("Task:", task.name)
print("Task type:", task_type)


## Load data

Load the selected split and optionally the matching few-shot examples. For `cot-N-shot`, examples are loaded from `example_cot/`; for `direct-N-shot`, they are loaded from `example/`.


In [ ]:
path_file_data = Path(args.dir_dataset[0]) / f"{task_name}.SFT.json"
list_dict_data = load_json(path_file_data)
list_dict_data = [x for x in list_dict_data if x["split"] == split]
if not list_dict_data:
    raise ValueError(f"No samples found for task={task_name}, split={split}")
print(f"Loaded {len(list_dict_data)} samples from {path_file_data}")
pprint(list_dict_data[0])


In [ ]:
# Optional few-shot examples.
prompt_base, num_example = parse_prompt_mode(prompt_mode)
if num_example is not None:
    example_dir = "example_cot" if prompt_base == "cot" else "example"
    path_file_example = Path(args.dir_dataset[0]) / example_dir / f"{task_name}.example.json"
    list_dict_example = load_json(path_file_example)[:num_example]
else:
    path_file_example = None
    list_dict_example = []

print("example file:", path_file_example)
print("examples:", len(list_dict_example))
if list_dict_example:
    pprint(list_dict_example[0])


## Client

Construct the Azure client from `configs/API.key.yaml`. This validates local configuration only; the API is contacted only when `RUN_SINGLE_QUERY = True`.


In [ ]:
client = create_azure_client(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=API_VERSION,
    api_key=AZURE_OPENAI_API_KEY,
)
client


## Single query

Use one sample to inspect the exact request and optionally send a minimal real request to Azure. Keep `RUN_SINGLE_QUERY = False` unless you intentionally want to spend API quota.


In [ ]:
idx = 0
input_one = list_dict_data[idx]
query_preview = {
    "instruction": input_one["instruction"],
    "input": input_one["input"],
}
pprint(query_preview)


In [ ]:
RUN_SINGLE_QUERY = False
if RUN_SINGLE_QUERY:
    if client is None:
        raise RuntimeError("Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY first.")
    result = azure_query_single(
        client=client,
        input=input_one,
        examples=list_dict_example,
        model_name=model_name,
        mode=mode,
        max_token_output=max_token_output,
    )
    pprint(result["params"])
    print(result["output"]["text"])
else:
    print("Set RUN_SINGLE_QUERY = True to call Azure OpenAI for one sample.")


## Batch query from in-memory data

Build a small preview payload without writing files or contacting Azure. This is the safest way to verify message order, custom ids, prompt mode, and token budget.


In [ ]:
preview_size = min(3, len(list_dict_data))
list_dict_data_batch = azure_query_batch_from_data(
    task_name=task_name,
    model_name=model_name,
    prompt_mode=prompt_mode,
    split=split,
    dataset=list_dict_data[:preview_size],
    examples=list_dict_example,
    mode=mode,
    max_token_input=max_token_input,
    max_token_output=max_token_output,
)
print(len(list_dict_data_batch))
pprint(list_dict_data_batch[0])


## Batch query from task

This writes the full task-level Azure batch JSONL to `azure/input/<model>/<prompt_mode>/`. Keep the flag off until the preview payload above looks correct.


In [ ]:
RUN_CREATE_TASK_BATCH = False
if RUN_CREATE_TASK_BATCH:
    list_dict_data_batch = azure_query_batch_from_task(
        task_name=task_name,
        model_name=model_name,
        prompt_mode=prompt_mode,
        split=split,
        mode=mode,
        path_dir_raw=args.dir_dataset[0],
        path_dir_batch="azure/input",
        max_token_input=max_token_input,
        max_token_output=max_token_output,
    )
    print(f"Created {len(list_dict_data_batch)} batch records")
else:
    print("Set RUN_CREATE_TASK_BATCH = True to write the task batch JSONL file.")


## Batch query from explicit files


In [ ]:
RUN_CREATE_FILE_BATCH = False
if RUN_CREATE_FILE_BATCH:
    path_file_result = f"azure/input/{model_name}/{prompt_mode}/{task_name}.batch.jsonl"
    list_dict_data_batch = azure_query_batch_from_file(
        task_name=task_name,
        model_name=model_name,
        prompt_mode=prompt_mode,
        split=split,
        mode=mode,
        path_file_data=str(path_file_data),
        path_file_example=str(path_file_example) if path_file_example else None,
        path_file_result=path_file_result,
        max_token_input=max_token_input,
        max_token_output=max_token_output,
    )
    print(f"Saved {len(list_dict_data_batch)} records to {path_file_result}")
else:
    print("Set RUN_CREATE_FILE_BATCH = True to write batch JSONL from explicit files.")


## Merge batch files


In [ ]:
RUN_MERGE_BATCH = False
if RUN_MERGE_BATCH:
    merge_azure_query_batch(
        model_name=model_name,
        prompt_mode=prompt_mode,
        path_dir_raw="azure/input",
        path_dir_merged="azure/input/merged",
        max_lines=50000,
        max_size_mb=200,
    )
else:
    print("Set RUN_MERGE_BATCH = True to merge batch files.")

## Parse Azure batch result

After downloading Azure batch output JSONL files to `azure/output/`, convert them back to the benchmark result layout under `result/<task>/<model>/`.


In [ ]:
# After downloading Azure batch output JSONL files, parse them into benchmark result rows.
path_dir_azure_result = "azure/output"

RUN_PARSE_BATCH_RESULT = False
if RUN_PARSE_BATCH_RESULT:
    transform_azure_result_to_task_result(
        path_dir_azure_result=path_dir_azure_result,
        path_dir_task_save=args.dir_result_save,
        path_dir_raw=args.dir_dataset[0],
        path_dir_batch="azure/input",
        decoding_strategy="greedy",
        seed=args.seed,
    )
else:
    print("Set RUN_PARSE_BATCH_RESULT = True after downloading Azure batch output files.")


## Cost estimation

Estimate batch cost from a sample of task inputs and an expected output length. This is only an estimate; final billing should use Azure usage metadata.


In [ ]:
RUN_COST_ESTIMATE = False
if RUN_COST_ESTIMATE:
    sample_inputs = [row["input"] for row in list_dict_data[:100]]
    estimated_cost = cost_estimation_word(
        list_input=sample_inputs,
        word_output_each=256,
        num_sample=len(list_dict_data),
        model_name=model_name,
        flag_batch=True,
    )
    print(f"Estimated batch cost: ${estimated_cost:.4f}")
else:
    print("Set RUN_COST_ESTIMATE = True to estimate batch cost from sample inputs.")
